# Returns Consumer - Quickstart

Exemplo mínimo mostrando como ler a tabela `returns` e calcular retorno acumulado usando pandas. Use este notebook como referência para consumidores (notebooks/modelagem).

## Observação sobre caminhos
O notebook tenta detectar automaticamente a raiz do repositório examinando
diretórios ascendentes à procura de `pyproject.toml` ou `.git`. Isso garante
que o arquivo de banco de dados seja localizado independentemente do diretório
de trabalho atual do kernel.


In [ ]:
import sqlite3
from pathlib import Path
from typing import Optional

import pandas as pd

# garante que estamos a olhar para o repositório (normalmente o cwd é a raiz)

def find_repo_root(start: Optional[Path] = None):
    """walk upwards until we find a marker of project root.

    notebooks are often executed from within their own folder in CI,
    so the start path may be `docs/examples/notebooks` rather than
    the repository top. look for `pyproject.toml` or a `.git`
    directory and return the containing path.
    """
    if start is None:
        start = Path.cwd()
    for p in (start, *start.parents):
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    raise FileNotFoundError("could not locate repository root")

repo_root = find_repo_root()

db_path = repo_root / "dados" / "data.db"

if not db_path.exists():
    raise FileNotFoundError(
        f"Database not found: {db_path!r}. "
        "Crie-o com `poetry run main` ou ajuste o caminho conforme necessário."
    )

conn = sqlite3.connect(db_path)
ticker = "PETR4.SA"
try:
    df = pd.read_sql_query(
        # historic schema uses ``return_value``; alias it for convenience
        "SELECT date, return_value AS return FROM returns ",
        "WHERE ticker = ? ORDER BY date",
        conn,
        params=(ticker,),
        parse_dates=["date"],
    )
    df = df.set_index("date")
    df["cumulative_return"] = (1 + df["return"]).cumprod() - 1
    print(df.tail())
except Exception as e:
    print(f"Failed to query database: {e}")
finally:
    conn.close()

            return  cumulative_return
date                                 
2026-02-26     0.1                0.1
